### Identifying Nearest Neighbors for Observation

In [15]:
from sklearn import datasets
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data

# Create a StandardScaler object
standardizer = StandardScaler()

# Standardize the features
features_standardized = standardizer.fit_transform(features)

# Create a NearestNeighbors object and fit it to the standardized features
nearest_neighbors = NearestNeighbors(n_neighbors=2).fit(features_standardized)

# Create a new observation
new_observation = [1, 1, 1, 1]

# Find the distances and indices of the nearest neighbors of the observation
distances, indices = nearest_neighbors.kneighbors([new_observation])

# Display the nearest neighbors
print(features_standardized[indices])

nearest_neighbors_euclidean = NearestNeighbors(
    n_neighbors=2, 
    metric='euclidean').fit(features_standardized)

print(nearest_neighbors_euclidean.kneighbors([new_observation]))

nearest_neighbors_manhattan = NearestNeighbors(n_neighbors=2, metric="manhattan").fit(
    features_standardized
)
print(nearest_neighbors_manhattan.kneighbors([new_observation]))

nearest_neighbors_minkowski = NearestNeighbors(n_neighbors=2, metric="minkowski").fit(
    features_standardized
)
print(nearest_neighbors_minkowski.kneighbors([new_observation]))


[[[1.03800476 0.55861082 1.10378283 1.18556721]
  [0.79566902 0.32841405 0.76275827 1.05393502]]]
(array([[0.49140089, 0.74294782]]), array([[124, 110]]))
(array([[0.76874397, 1.16709368]]), array([[124, 110]]))
(array([[0.49140089, 0.74294782]]), array([[124, 110]]))


### Creating a k-nearest neighbors classifier

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn import datasets

# Load the iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Create a StandardScaler object
standardizer = StandardScaler()

# Standardize the features
X_std = standardizer.fit_transform(X)

# Train a KNN classifier using 5 neighbors
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1).fit(X_std, y)

# Create two new observations
new_observations = [[0.75, 0.75, 0.75, 0.75],
                    [1, 1, 1, 1]]

# Predict the classes of the new observations
print(knn.predict(new_observations))

# Predict the probabilities of the new observations
print(knn.predict_proba(new_observations))

[1 2]
[[0.  0.6 0.4]
 [0.  0.  1. ]]


### Determining the optimal number of nearest neighbors

In [18]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import GridSearchCV

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Create a StandardScaler object
standardizer = StandardScaler()

# Standardize the features
features_standardized = standardizer.fit_transform(features)

# Create a KNN classifier
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

# Create a pipeline that standardizes the features and then applies the KNN classifier
pipe = Pipeline([("standardizer", standardizer), ("knn", knn)])

# Create the search space for hyperparameter tuning
search_space = [{"knn__n_neighbors": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}]

# Организовамь поиск по семке
classifier = GridSearchCV(pipe, search_space, cv=5, verbose=0).fit(
    features_standardized, target
)

# Best hyperparameter value
classifier.best_estimator_.get_params()["knn__n_neighbors"]

6

### Creating a classifier using the nearest neighbors method within a given radius

In [19]:
from sklearn.neighbors import RadiusNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn import datasets

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Create a StandardScaler object
standardizer = StandardScaler()

# Standardize the features
features_standardized = standardizer.fit_transform(features)

# Train a radius-based nearest neighbors classifier
rnn = RadiusNeighborsClassifier(radius=0.5, n_jobs=-1).fit(
    features_standardized, target
)

# Create two new observations
new_observations = [[1, 1, 1, 1]]

# Predict the class of the new observations
rnn.predict(new_observations)


array([2])

### Search for closest neighbors

In [24]:
import faiss
import numpy as np
from sklearn import datasets
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data

# Create a StandardScaler object
standardizer = StandardScaler()

# Standardize the features
features_standardized = standardizer.fit_transform(features)

# Насмроимь парамемры faiss
n_features = features_standardized.shape[1]
nlist = 3
k = 2

# Create a FAISS index for approximate nearest neighbor search
quantizer = faiss.IndexFlatIP(n_features)
index = faiss.IndexIVFFlat(quantizer, n_features, nlist)

# Train the index and add the standardized features
index.train(features_standardized)
index.add(features_standardized)

# Create a new observation
new_observation = np.array([[1, 1, 1, 1]])

# Perform a search for the two nearest neighbors in the index
distances, indices = index.search(new_observation, k)

# Print the feature vectors of the two nearest neighbors
np.array([list(features_standardized[i]) for i in indices[0]])

array([[1.03800476, 0.55861082, 1.10378283, 1.18556721],
       [0.79566902, 0.32841405, 0.76275827, 1.05393502]])

### Evaluation of the approximate nearest neighbors method

In [27]:
import faiss
import numpy as np
from sklearn import datasets
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Number of nearest neighbors
k = 10

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data

# Create a StandardScaler object
standardizer = StandardScaler()

# Standardize the features
features_standardized = standardizer.fit_transform(features)

# Create a NearestNeighbors object and fit it to the standardized features
nearest_neighbors = NearestNeighbors(n_neighbors=k).fit(features_standardized)

# Adjust the number of features for FAISS
n_features = features_standardized.shape[1]

# Create a FAISS index for approximate nearest neighbor search
quantizer = faiss.IndexFlatIP(n_features)
index = faiss.IndexIVFFlat(quantizer, n_features, nlist)

# Train the index and add the standardized features
index.train(features_standardized)
index.add(features_standardized)
index.nprobe = 1

# Create a new observation
new_observation = np.array([[1, 1, 1, 1]])

# Predict the distances and indices of the nearest neighbors for the new observation
knn_distances, knn_indices = nearest_neighbors.kneighbors(new_observation)

# Predict the distances and indices of the nearest neighbors for the new observation using FAISS
ivf_distances, ivf_indices = index.search(new_observation, k)

# Find the intersection of the indices returned by KNN and FAISS
recalled_items = set(list(knn_indices[0])) & set(list(ivf_indices[0]))

# Print the recall at k
print(f"Recall @k={k}: {len(recalled_items)/k * 100}%")


Recall @k=10: 100.0%
